# microWakeWord Easy Training Notebook

This notebook guides you through training a custom wake word for your HAVoice PE device.

## Requirements
- Python 3.10
- TensorFlow 2.15+
- 8GB+ RAM (16GB recommended)

## Quick Start
1. Configure your wake word in the next cell
2. Run all cells (Shift+Enter or Run All)
3. Wait for training to complete
4. Download the model files from the output directory

## Step 1: Configure Your Wake Word

In [ ]:
# =============================================================================
# CONFIGURE YOUR WAKE WORD HERE
# =============================================================================

# Your wake word phrase (use underscores for spaces)
WAKE_WORD = "hey_freya"

# Display name for the wake word
DISPLAY_NAME = "Hey Freya"

# Output directory for the trained model
OUTPUT_DIR = "../models/hey_freya"

# =============================================================================
# TRAINING SETTINGS
# =============================================================================

# Preset based on wake word length:
# - "short": 1-2 syllables (e.g., "hey bot")
# - "medium": 3-4 syllables (e.g., "hey freya") [RECOMMENDED]
# - "long": 5+ syllables (e.g., "hey computer assistant")
PRESET = "medium"

# Number of synthetic samples to generate
# More samples = better accuracy but longer training
# Recommended: 5000-20000
NUM_SAMPLES = 10000

# Augmentation level: "light", "medium", "heavy"
# Heavy augmentation helps with noise robustness
AUGMENTATION = "medium"

# =============================================================================
# PERSONAL SAMPLES (Optional but recommended)
# =============================================================================

# Set to True if you have personal voice recordings
USE_PERSONAL_SAMPLES = True

# Path to your personal voice samples (WAV files)
PERSONAL_SAMPLES_DIR = "../samples/personal"

# =============================================================================
# DETECTION THRESHOLDS
# =============================================================================

# Probability cutoff (0.5-0.95)
# Lower = more sensitive, Higher = fewer false positives
PROBABILITY_CUTOFF = 0.85

# Sliding window size for detection stability
SLIDING_WINDOW_SIZE = 5

print(f"Configuration loaded for wake word: '{DISPLAY_NAME}'")

## Step 2: Install Dependencies

In [ ]:
import subprocess
import sys

def install_if_missing(package, import_name=None):
    import_name = import_name or package
    try:
        __import__(import_name)
        print(f"  {package} is installed")
    except ImportError:
        print(f"  Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

print("Checking dependencies...")
install_if_missing("tensorflow")
install_if_missing("numpy")
install_if_missing("scipy")
install_if_missing("librosa")
install_if_missing("soundfile")
install_if_missing("pyyaml", "yaml")
install_if_missing("tqdm")
install_if_missing("matplotlib")
print("\nAll dependencies installed!")

## Step 3: Import Libraries and Setup

In [ ]:
import os
import json
import random
import numpy as np
import tensorflow as tf
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "logs"), exist_ok=True)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")
print(f"Output directory: {os.path.abspath(OUTPUT_DIR)}")

## Step 4: Audio Processing Utilities

In [ ]:
import librosa
import soundfile as sf
from scipy import signal

# Audio parameters matching microWakeWord requirements
SAMPLE_RATE = 16000
FRAME_LENGTH = 480  # 30ms at 16kHz
FRAME_STEP = 160    # 10ms at 16kHz
NUM_MEL_BINS = 40
FEATURE_LENGTH = 198  # ~2 seconds of audio

def load_audio(file_path, target_sr=SAMPLE_RATE):
    """Load and resample audio file."""
    audio, sr = librosa.load(file_path, sr=target_sr, mono=True)
    return audio

def extract_features(audio, sr=SAMPLE_RATE):
    """Extract mel spectrogram features matching microWakeWord format."""
    # Compute mel spectrogram
    mel_spec = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_fft=512,
        hop_length=FRAME_STEP,
        n_mels=NUM_MEL_BINS,
        fmin=60,
        fmax=3800
    )
    
    # Convert to log scale
    log_mel = librosa.power_to_db(mel_spec, ref=np.max)
    
    # Normalize
    log_mel = (log_mel - log_mel.mean()) / (log_mel.std() + 1e-8)
    
    return log_mel.T  # Transpose to (time, features)

def pad_or_truncate(features, target_length=FEATURE_LENGTH):
    """Ensure features are the correct length."""
    if len(features) > target_length:
        # Random crop
        start = random.randint(0, len(features) - target_length)
        return features[start:start + target_length]
    elif len(features) < target_length:
        # Pad with zeros
        padding = np.zeros((target_length - len(features), features.shape[1]))
        return np.vstack([features, padding])
    return features

print("Audio processing utilities loaded!")

## Step 5: Data Augmentation

In [ ]:
class AudioAugmenter:
    """Audio augmentation for wake word training."""
    
    def __init__(self, level="medium"):
        self.level = level
        self.settings = {
            "light": {
                "noise_range": (0.001, 0.005),
                "speed_range": (0.95, 1.05),
                "pitch_range": (-1, 1),
                "volume_range": (0.9, 1.1)
            },
            "medium": {
                "noise_range": (0.002, 0.01),
                "speed_range": (0.9, 1.1),
                "pitch_range": (-2, 2),
                "volume_range": (0.7, 1.3)
            },
            "heavy": {
                "noise_range": (0.005, 0.02),
                "speed_range": (0.85, 1.15),
                "pitch_range": (-3, 3),
                "volume_range": (0.5, 1.5)
            }
        }[level]
    
    def add_noise(self, audio):
        """Add random noise."""
        noise_level = random.uniform(*self.settings["noise_range"])
        noise = np.random.randn(len(audio)) * noise_level
        return audio + noise
    
    def change_speed(self, audio):
        """Change playback speed."""
        speed = random.uniform(*self.settings["speed_range"])
        return librosa.effects.time_stretch(audio, rate=speed)
    
    def change_pitch(self, audio, sr=SAMPLE_RATE):
        """Change pitch."""
        pitch = random.uniform(*self.settings["pitch_range"])
        return librosa.effects.pitch_shift(audio, sr=sr, n_steps=pitch)
    
    def change_volume(self, audio):
        """Change volume."""
        volume = random.uniform(*self.settings["volume_range"])
        return audio * volume
    
    def augment(self, audio):
        """Apply random augmentations."""
        if random.random() < 0.7:
            audio = self.add_noise(audio)
        if random.random() < 0.5:
            audio = self.change_speed(audio)
        if random.random() < 0.5:
            audio = self.change_pitch(audio)
        if random.random() < 0.7:
            audio = self.change_volume(audio)
        return audio

augmenter = AudioAugmenter(AUGMENTATION)
print(f"Augmenter initialized with level: {AUGMENTATION}")

## Step 6: Generate/Load Training Samples

In [ ]:
def generate_synthetic_samples(wake_word, num_samples, use_tts=True):
    """
    Generate synthetic training samples.
    
    In production, this uses Piper TTS to generate diverse voice samples.
    For this notebook, we'll create placeholder data that demonstrates the structure.
    
    For real training, you should:
    1. Install Piper TTS: pip install piper-tts
    2. Download voice models from: https://github.com/rhasspy/piper/releases
    3. Generate samples with varying voices, speeds, and pitches
    """
    print(f"\nGenerating {num_samples} synthetic samples for '{wake_word}'...")
    print("(Using simulated data - for production, integrate Piper TTS)")
    
    samples = []
    
    # Simulate sample generation
    for i in tqdm(range(num_samples), desc="Generating"):
        # Create synthetic audio (replace with actual TTS in production)
        duration = random.uniform(0.8, 1.5)  # Variable duration
        t = np.linspace(0, duration, int(SAMPLE_RATE * duration))
        
        # Simple synthetic "speech-like" signal
        freq = random.uniform(100, 300)  # Fundamental frequency
        audio = np.sin(2 * np.pi * freq * t) * np.exp(-t * 2)
        
        # Add harmonics
        for h in [2, 3, 4]:
            audio += 0.3/h * np.sin(2 * np.pi * freq * h * t) * np.exp(-t * 2)
        
        # Add envelope
        envelope = np.concatenate([
            np.linspace(0, 1, int(len(audio) * 0.1)),
            np.ones(int(len(audio) * 0.6)),
            np.linspace(1, 0, len(audio) - int(len(audio) * 0.7))
        ])
        audio = audio * envelope[:len(audio)]
        
        # Normalize
        audio = audio / (np.max(np.abs(audio)) + 1e-8)
        
        # Apply augmentation
        audio = augmenter.augment(audio)
        
        samples.append(audio)
    
    return samples

def load_personal_samples(samples_dir):
    """Load personal voice samples from directory."""
    samples = []
    samples_path = Path(samples_dir)
    
    if not samples_path.exists():
        print(f"Personal samples directory not found: {samples_dir}")
        return samples
    
    wav_files = list(samples_path.glob("*.wav"))
    print(f"\nLoading {len(wav_files)} personal samples...")
    
    for wav_file in tqdm(wav_files, desc="Loading"):
        try:
            audio = load_audio(wav_file)
            samples.append(audio)
            
            # Generate augmented versions
            for _ in range(5):
                augmented = augmenter.augment(audio.copy())
                samples.append(augmented)
        except Exception as e:
            print(f"Error loading {wav_file}: {e}")
    
    return samples

# Generate/load positive samples
positive_samples = generate_synthetic_samples(WAKE_WORD, NUM_SAMPLES)

if USE_PERSONAL_SAMPLES:
    personal = load_personal_samples(PERSONAL_SAMPLES_DIR)
    if personal:
        print(f"Added {len(personal)} personal samples (including augmented)")
        positive_samples.extend(personal)

print(f"\nTotal positive samples: {len(positive_samples)}")

## Step 7: Generate Negative Samples

In [ ]:
def generate_negative_samples(num_samples):
    """
    Generate negative (non-wake-word) samples.
    
    For production, use:
    1. Pre-generated negatives from HuggingFace
    2. Common speech that sounds similar to wake word
    3. Background noise, music, TV audio
    """
    print(f"\nGenerating {num_samples} negative samples...")
    
    samples = []
    
    for i in tqdm(range(num_samples), desc="Generating"):
        duration = random.uniform(1.0, 2.0)
        num_points = int(SAMPLE_RATE * duration)
        
        sample_type = random.choice(["noise", "speech", "music", "silence"])
        
        if sample_type == "noise":
            # Background noise
            audio = np.random.randn(num_points) * random.uniform(0.01, 0.1)
            
        elif sample_type == "speech":
            # Simulated speech (different from wake word)
            t = np.linspace(0, duration, num_points)
            freq = random.uniform(80, 400)
            audio = np.sin(2 * np.pi * freq * t) * 0.3
            audio += np.random.randn(num_points) * 0.02
            
        elif sample_type == "music":
            # Simulated music (multiple tones)
            t = np.linspace(0, duration, num_points)
            audio = np.zeros(num_points)
            for _ in range(5):
                freq = random.uniform(200, 2000)
                audio += np.sin(2 * np.pi * freq * t) * random.uniform(0.1, 0.3)
                
        else:  # silence
            audio = np.random.randn(num_points) * 0.001
        
        # Normalize
        audio = audio / (np.max(np.abs(audio)) + 1e-8) * random.uniform(0.1, 0.8)
        samples.append(audio)
    
    return samples

negative_samples = generate_negative_samples(NUM_SAMPLES)
print(f"\nTotal negative samples: {len(negative_samples)}")

## Step 8: Prepare Training Dataset

In [ ]:
def prepare_dataset(positive_samples, negative_samples):
    """Convert audio samples to features and create training dataset."""
    print("\nPreparing dataset...")
    
    X = []
    y = []
    
    # Process positive samples
    print("Processing positive samples...")
    for audio in tqdm(positive_samples, desc="Positive"):
        features = extract_features(audio)
        features = pad_or_truncate(features)
        X.append(features)
        y.append(1)
    
    # Process negative samples
    print("Processing negative samples...")
    for audio in tqdm(negative_samples, desc="Negative"):
        features = extract_features(audio)
        features = pad_or_truncate(features)
        X.append(features)
        y.append(0)
    
    X = np.array(X)
    y = np.array(y)
    
    # Shuffle
    indices = np.random.permutation(len(X))
    X = X[indices]
    y = y[indices]
    
    # Split into train/validation
    split_idx = int(len(X) * 0.8)
    X_train, X_val = X[:split_idx], X[split_idx:]
    y_train, y_val = y[:split_idx], y[split_idx:]
    
    print(f"\nDataset prepared:")
    print(f"  Training: {len(X_train)} samples")
    print(f"  Validation: {len(X_val)} samples")
    print(f"  Feature shape: {X_train[0].shape}")
    
    return X_train, X_val, y_train, y_val

X_train, X_val, y_train, y_val = prepare_dataset(positive_samples, negative_samples)

## Step 9: Build Model Architecture

In [ ]:
def build_model(input_shape, preset="medium"):
    """
    Build wake word detection model.
    
    Architecture based on microWakeWord's streaming CNN design,
    optimized for low-power microcontrollers.
    """
    model_configs = {
        "small": {"filters": [16, 32, 48], "dense": 64},
        "medium": {"filters": [24, 48, 64], "dense": 96},
        "large": {"filters": [32, 64, 96], "dense": 128}
    }
    
    config = model_configs[preset]
    
    model = tf.keras.Sequential([
        # Input layer
        tf.keras.layers.Input(shape=input_shape),
        
        # Add channel dimension for Conv2D
        tf.keras.layers.Reshape((input_shape[0], input_shape[1], 1)),
        
        # First conv block
        tf.keras.layers.Conv2D(config["filters"][0], (3, 3), padding="same"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Dropout(0.25),
        
        # Second conv block
        tf.keras.layers.Conv2D(config["filters"][1], (3, 3), padding="same"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Dropout(0.25),
        
        # Third conv block
        tf.keras.layers.Conv2D(config["filters"][2], (3, 3), padding="same"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Dropout(0.25),
        
        # Global pooling and dense layers
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(config["dense"], activation="relu"),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
    )
    
    return model

# Build model
input_shape = (FEATURE_LENGTH, NUM_MEL_BINS)
model = build_model(input_shape, preset=PRESET)

print(f"\nModel built with preset: {PRESET}")
model.summary()

## Step 10: Train the Model

In [ ]:
# Callbacks
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6
    ),
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_DIR, "best_model.keras"),
        monitor="val_loss",
        save_best_only=True
    ),
    tf.keras.callbacks.TensorBoard(
        log_dir=os.path.join(OUTPUT_DIR, "logs")
    )
]

print("Starting training...")
print("="*50)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

print("\nTraining complete!")

## Step 11: Evaluate and Visualize Results

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Loss
axes[0, 0].plot(history.history["loss"], label="Training")
axes[0, 0].plot(history.history["val_loss"], label="Validation")
axes[0, 0].set_title("Loss")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].legend()

# Accuracy
axes[0, 1].plot(history.history["accuracy"], label="Training")
axes[0, 1].plot(history.history["val_accuracy"], label="Validation")
axes[0, 1].set_title("Accuracy")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].legend()

# Precision
axes[1, 0].plot(history.history["precision"], label="Training")
axes[1, 0].plot(history.history["val_precision"], label="Validation")
axes[1, 0].set_title("Precision")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].legend()

# Recall
axes[1, 1].plot(history.history["recall"], label="Training")
axes[1, 1].plot(history.history["val_recall"], label="Validation")
axes[1, 1].set_title("Recall")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_history.png"))
plt.show()

# Final evaluation
print("\n" + "="*50)
print("Final Evaluation on Validation Set")
print("="*50)
results = model.evaluate(X_val, y_val, verbose=0)
print(f"Loss: {results[0]:.4f}")
print(f"Accuracy: {results[1]:.4f}")
print(f"Precision: {results[2]:.4f}")
print(f"Recall: {results[3]:.4f}")

## Step 12: Convert to TensorFlow Lite

In [ ]:
def convert_to_tflite(model, output_path, quantize=True):
    """Convert Keras model to TensorFlow Lite format."""
    print("\nConverting to TensorFlow Lite...")
    
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    
    if quantize:
        print("  Applying int8 quantization...")
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.int8]
        
        # Representative dataset for quantization
        def representative_dataset():
            for i in range(min(100, len(X_train))):
                yield [X_train[i:i+1].astype(np.float32)]
        
        converter.representative_dataset = representative_dataset
    
    tflite_model = converter.convert()
    
    # Save model
    tflite_path = os.path.join(output_path, f"{WAKE_WORD}.tflite")
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)
    
    model_size = os.path.getsize(tflite_path) / 1024
    print(f"  Model saved: {tflite_path}")
    print(f"  Model size: {model_size:.1f} KB")
    
    return tflite_path

tflite_path = convert_to_tflite(model, OUTPUT_DIR, quantize=True)

## Step 13: Create Model Manifest

In [ ]:
def create_manifest(wake_word, output_path, probability_cutoff, sliding_window_size):
    """Create JSON manifest for microWakeWord/ESPHome."""
    
    manifest = {
        "name": wake_word,
        "display_name": DISPLAY_NAME,
        "version": "1.0.0",
        "model_file": f"{wake_word}.tflite",
        "probability_cutoff": probability_cutoff,
        "sliding_window_size": sliding_window_size,
        "sample_rate": SAMPLE_RATE,
        "num_mel_bins": NUM_MEL_BINS,
        "frame_length_ms": 30,
        "frame_step_ms": 10,
        "created_with": "microWakeWord Easy Training Notebook",
        "training_config": {
            "preset": PRESET,
            "augmentation": AUGMENTATION,
            "num_samples": NUM_SAMPLES,
            "used_personal_samples": USE_PERSONAL_SAMPLES
        }
    }
    
    manifest_path = os.path.join(output_path, f"{wake_word}.json")
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)
    
    print(f"\nManifest saved: {manifest_path}")
    print(json.dumps(manifest, indent=2))
    
    return manifest_path

manifest_path = create_manifest(
    WAKE_WORD,
    OUTPUT_DIR,
    PROBABILITY_CUTOFF,
    SLIDING_WINDOW_SIZE
)

## Step 14: Generate ESPHome Configuration

In [ ]:
esphome_config = f"""# ESPHome microWakeWord Configuration for {DISPLAY_NAME}
# Generated by Easy Training Notebook
#
# Add this to your Voice PE ESPHome configuration

micro_wake_word:
  models:
    # Custom wake word: "{DISPLAY_NAME}"
    - model: /config/custom_wakewords/{WAKE_WORD}.json
      id: {WAKE_WORD}
      probability_cutoff: {PROBABILITY_CUTOFF}
      sliding_window_size: {SLIDING_WINDOW_SIZE}
    
    # Remove default wake words to save memory (optional)
    # - id: !remove hey_jarvis
    # - id: !remove alexa
    # - id: !remove hey_mycroft

# Voice assistant event handler for {DISPLAY_NAME}
voice_assistant:
  on_wake_word_detected:
    - logger.log: "Wake word '{DISPLAY_NAME}' detected!"
    - light.turn_on:
        id: led_ring
        effect: "Wake Word Pulse"
        brightness: 100%
        red: 0%
        green: 100%
        blue: 0%
"""

config_path = os.path.join(OUTPUT_DIR, f"{WAKE_WORD}_esphome.yaml")
with open(config_path, "w") as f:
    f.write(esphome_config)

print("ESPHome configuration snippet:")
print("="*50)
print(esphome_config)

## Step 15: Summary and Deployment Instructions

In [ ]:
print("="*60)
print(f"  TRAINING COMPLETE: {DISPLAY_NAME}")
print("="*60)
print()
print("Generated files:")
print(f"  - {WAKE_WORD}.tflite (model)")
print(f"  - {WAKE_WORD}.json (manifest)")
print(f"  - {WAKE_WORD}_esphome.yaml (ESPHome config snippet)")
print(f"  - training_history.png (training plots)")
print()
print("Output directory:")
print(f"  {os.path.abspath(OUTPUT_DIR)}")
print()
print("="*60)
print("  DEPLOYMENT INSTRUCTIONS")
print("="*60)
print()
print("1. Copy model files to Home Assistant:")
print(f"   cp {OUTPUT_DIR}/{WAKE_WORD}.tflite /config/custom_wakewords/")
print(f"   cp {OUTPUT_DIR}/{WAKE_WORD}.json /config/custom_wakewords/")
print()
print("2. Update your Voice PE ESPHome configuration:")
print(f"   Add the configuration from {WAKE_WORD}_esphome.yaml")
print()
print("3. Flash the Voice PE device:")
print("   ESPHome Dashboard -> Your Device -> Install")
print()
print("4. Select wake word in Home Assistant:")
print("   Settings -> Voice assistants -> Edit -> Wake word")
print()
print("="*60)
print()
print(f"Your wake word '{DISPLAY_NAME}' is ready!")